# NLP Interview Questions — from TF-IDF to the LLM Era

NLP rounds in 2026 are two rounds in one: the classical pipeline (still heavily tested — tokenization, TF-IDF, embeddings) and the LLM era (attention, fine-tuning, RAG, evaluation). This notebook executes both layers, ending with an LLM-question section calibrated to what interviewers actually ask about the current landscape.

## Q1 — Tokenization: the Pipeline's Hidden Foundation

**Word-level** (split on whitespace) — simple, explodes vocabulary, can't handle OOV/unseen morphologies. **Subword (BPE/WordPiece/SentencePiece)** — the modern default: frequent words stay whole, rare words decompose into learned pieces ("unbelievable" → "un" + "believ" + "able"); OOV becomes impossible because everything reduces to the learned subword inventory. **Character-level** — no OOV at all, but sequences get long and the model must learn morphology from scratch. Say: *"the tokenizer choice sets the vocabulary-vs-sequence-length tradeoff, and it's baked into the pretrained model — you don't choose it independently."*

In [1]:
# a byte-pair-encoding spirit demo: learn merges from data
from collections import Counter

corpus = ["low lower lowest", "slow slower slowest", "new newest newer new"]
word_freqs = Counter()
for sent in corpus:
    word_freqs.update(sent.split())

# pretend-merges: count common character pairs (real BPE iterates frequency-ranked merges)
def top_pairs(freqs):
    pairs = Counter()
    for w, f in freqs.items():
        chars = list(w)
        for a, b in zip(chars, chars[1:]):
            pairs[(a, b)] += f
    return pairs.most_common(5)

print("word frequencies:", dict(word_freqs))
print("top character pairs (merge candidates):", top_pairs(word_freqs))
print("BPE would merge 'e','r' and 'e','s' first — 'slowest' becomes slow+est after a few merges")

word frequencies: {'low': 1, 'lower': 1, 'lowest': 1, 'slow': 1, 'slower': 1, 'slowest': 1, 'new': 2, 'newest': 1, 'newer': 1}
top character pairs (merge candidates): [(('l', 'o'), 6), (('o', 'w'), 6), (('w', 'e'), 6), (('n', 'e'), 4), (('e', 'w'), 4)]
BPE would merge 'e','r' and 'e','s' first — 'slowest' becomes slow+est after a few merges


## Q2 — TF-IDF: Write the Formula, Feel the Intuition

**TF-IDF(t, d) = TF(t, d) × IDF(t)** where IDF(t) = log(N / df(t)) (+1 smearing in practice). Intuition to say: *"every term is weighted by how surprising it is across the corpus — a word in this document but rare elsewhere is informative; a word everywhere (the, and) carries ~no information."* It's a hand-crafted feature, not learned — that's both its strength (no training, interpretable) and its ceiling (no semantics: "car" ≠ "automobile" to TF-IDF).

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["the model trains on text data",
        "the model evaluates on text data",
        "gradient descent trains the model weights",
        "attention attends to relevant tokens"]

vec = TfidfVectorizer()
Z = vec.fit_transform(docs)
idf = pd.Series(vec.idf_, index=vec.get_feature_names_out(), name="idf")
print(idf.sort_values().round(2).to_string())
print("\n'the' has lowest IDF (in every doc); 'attention', 'gradient', 'relevant' highest")

model        1.22
the          1.22
on           1.51
data         1.51
text         1.51
trains       1.51
descent      1.92
attends      1.92
evaluates    1.92
gradient     1.92
attention    1.92
relevant     1.92
to           1.92
tokens       1.92
weights      1.92

'the' has lowest IDF (in every doc); 'attention', 'gradient', 'relevant' highest


## Q3 — Word Embeddings: from Distributional Hypothesis to Vectors

**The distributional hypothesis:** a word is characterized by the company it keeps. **word2vec** trains a shallow net so that words in similar contexts get similar vectors (skip-gram predicts context from word; CBOW the reverse). The analogy arithmetic (king − man + woman ≈ queen) made embeddings famous — because vector directions encode relations. **GloVe** factorizes a global co-occurrence matrix instead. Both are **static**: one vector per word, so "bank" is one point though it means riverbank and finance. **Contextual embeddings** (BERT, GPT, modern LLMs) compute the vector *per occurrence* — "bank" in "river bank" and "bank loan" get different vectors. That static→contextual jump is the single most important NLP interview sentence.

In [3]:
# the analogy property, demonstrated on a tiny trained-by-counting space
# co-occurrence rows = naive embeddings; cosine geometry still shows relations
counts = pd.DataFrame({
    "water": [5, 0, 1, 0], "money": [0, 5, 1, 0], "trees": [4, 0, 0, 1],
}, index=["river", "bank_finance", "bank_ambiguous", "forest"])

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("cos(river, forest)         :", round(cos(counts.loc['river'], counts.loc['forest']), 3))
print("cos(bank_finance, river)   :", round(cos(counts.loc['bank_finance'], counts.loc['river']), 3))
print("static embedding would force ONE 'bank' vector = average of both senses")
print("contextual models avoid this by construction — the vector depends on neighbors")

cos(river, forest)         : 0.625
cos(bank_finance, river)   : 0.0
static embedding would force ONE 'bank' vector = average of both senses
contextual models avoid this by construction — the vector depends on neighbors


## Q4 — Attention: the Equation, Executed and Explained

**Attention(Q, K, V) = softmax(QKᵀ/√d)·V.** The story to tell: each token emits a **query** ("what am I looking for?"), a **key** ("what do I offer?"), and a **value** ("what I actually pass along"). QKᵀ scores every query against every key; softmax turns scores into a mixing distribution; the output is the value-mix. The **√d** scaling keeps softmax gradients healthy as dimension grows (dot products grow with d; unscaled → saturated softmax → dead gradients). **Multi-head**: h parallel attention runs in lower dimension, each head free to specialize (syntax, coreference, position); concat + project.

In [4]:
rng = np.random.default_rng(0)
d, T = 16, 6
words = ["the", "model", "attends", "to", "tokens", "now"]
Q, K, V = (rng.normal(size=(T, d)) * 0.5 for _ in range(T and 3))

scores = Q @ K.T / np.sqrt(d)
weights = np.exp(scores) / np.exp(scores).sum(-1, keepdims=True)
out = weights @ V

att = pd.DataFrame(weights.round(2), index=words, columns=words)
print("attention weights (rows=queries, cols=keys):")
print(att.to_string())
print("\neach row is a probability distribution — how much that query token mixes each key's value")

attention weights (rows=queries, cols=keys):
          the  model  attends    to  tokens   now
the      0.22   0.18     0.14  0.19    0.09  0.17
model    0.16   0.19     0.20  0.18    0.14  0.14
attends  0.12   0.20     0.20  0.13    0.18  0.17
to       0.17   0.18     0.18  0.17    0.17  0.12
tokens   0.16   0.24     0.13  0.12    0.23  0.12
now      0.16   0.16     0.23  0.14    0.13  0.18

each row is a probability distribution — how much that query token mixes each key's value


## Q5 — the Pretraining → Fine-tuning Ladder

The paradigm in one line each:
- **Pretraining** — self-supervised on massive text (next-token for GPT-style causal LMs; masked-token for BERT-style encoders): learns language, facts, reasoning patterns.
- **Supervised fine-tuning (SFT)** — instruction-response pairs teach the *format* of helpfulness.
- **Parameter-efficient fine-tuning** — **LoRA**: freeze the base, train low-rank adapters (ΔW = BA, rank r ≪ d); ~0.1–1% of parameters trained, adapters hot-swappable per task. **QLoRA**: LoRA on a quantized base — 70B fine-tuning on one GPU.
- **Preference tuning** — RLHF/DPO align outputs with human preferences (DPO is the simpler "classify preferred over rejected" objective — name it as the 2026 default).
- **Prompting / in-context learning** — zero training; the "programming language" view of LLMs.

**When to fine-tune vs RAG?** The decision table interviewers want: *fine-tune for style/format/domain behavior; RAG for knowledge that changes or must be cited.* Most production systems do RAG first, fine-tune later, on top.

## Q6 — RAG, Structured (Because Every Round Asks It Now)

Retrieval-Augmented Generation, the six components with the design choice inside each:

1. **Chunking** — fixed-size (~256–512 tokens) vs structure-aware (by heading/paragraph); overlap windows to protect boundary sentences.
2. **Embedding model** — off-the-shelf sentence embedders vs domain-trained; must be the *same family* for query and documents (bi-encoders encode separately — that's what makes retrieval fast and scalable).
3. **Vector index** — exact (flat, small corpora) vs ANN (HNSW / IVF: sub-linear recall-vs-speed tradeoff).
4. **Retrieval strategy** — hybrid (BM25 + dense, fused with RRF — *say hybrid*, it's the standard answer), top-k 5–20, optional reranker (cross-encoder scores query-doc jointly — more accurate, slower — used to rerank the top 50).
5. **Generation** — the context goes in the prompt; citations = chunk provenance.
6. **Evaluation** — retrieval metrics (recall@k, MRR) *and* end-to-end (faithfulness, answer relevance; RAGAS-style frameworks).

The [LLM and RAG System Design notebook](LLM%20and%20RAG%20System%20Design.ipynb) does the capacity math; here it's the conceptual layer.

In [5]:
# hybrid retrieval intuition: BM25 handles rare keywords, dense handles paraphrase
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = ["gradient descent optimizes neural weights",
        "the cat sat on the mat",
        "optimizer updates parameters with learning rates",
        "cats like warm mats and sunny spots"]
query = ["how are neural network parameters optimized?"]

# lexical: BM25-spirit via TF-IDF
tfidf = TfidfVectorizer().fit(docs + query)
lex = cosine_similarity(tfidf.transform(query), tfidf.transform(docs))[0]

# dense-spirit: character n-grams stand in for a real embedder (paraphrase-ish)
from sklearn.feature_extraction.text import TfidfVectorizer as TV
char_vec = TV(analyzer="char_wb", ngram_range=(3, 5)).fit(docs + query)
sem = cosine_similarity(char_vec.transform(query), char_vec.transform(docs))[0]

hybrid = pd.DataFrame({"lexical": lex.round(3), "char-semantic": sem.round(3)},
                      index=["desc: gradient descent", "cat mat", "optimizer params", "cats warm mats"])
hybrid["fused"] = (hybrid.rank() .mean(axis=1)).round(2)
print(hybrid.to_string())
print("query matches 'optimizer parameters' best lexically AND semantically — hybrid fuses both signals")

                        lexical  char-semantic  fused
desc: gradient descent    0.131          0.229    3.5
cat mat                   0.000          0.000    1.5
optimizer params          0.119          0.292    3.5
cats warm mats            0.000          0.000    1.5
query matches 'optimizer parameters' best lexically AND semantically — hybrid fuses both signals


## Q7 — LLM-Era Questions, the 2026 Set

- **Why do LLMs hallucinate?** They generate the most plausible next token, not the most *verified* one; nothing in pretraining enforces grounding. Mitigations: RAG grounding, lower temperature for factual tasks, citation requirements, self-consistency checks.
- **Context window vs RAG?** Long contexts make more fit in-prompt but cost quadratic attention compute and show "lost in the middle" degradation; RAG remains the answer for corpora that don't fit or change daily.
- **Temperature/top-p?** Sampling controls: temperature flattens/sharpens the distribution; top-p (nucleus) samples from the smallest set of tokens whose cumulative prob ≥ p. For factual QA: low temperature or greedy.
- **How do you evaluate an LLM system?** Task metrics (accuracy against gold), faithfulness/groundedness for RAG, LLM-as-judge for open-ended (with its own reliability checks — say it, the meta-evaluation caveat), latency/cost per query, and red-team passes for safety.
- **Quantization?** Weights to int8/int4 (GPTQ/AWQ families; QLoRA for training-time) — 2–4x memory cuts at small quality cost; the production default for self-hosted inference.
- **Agents?** LLM + tools + loop (plan → act → observe); the reliability question is the interview: function-calling accuracy, error recovery, budgeted step limits, and evaluation harnesses. Mention the Microsoft Agent Framework / LangGraph ecosystem as the tooling layer.

## Q8 — Classical NLP You're Still Expected to Know

**POS tagging / NER / parsing** (sequence labeling — BIO tags, CRFs historically, transformers now). **Lemmatization vs stemming** — dictionary-backed canonical form vs crude suffix chopping ("studies"→"study" vs "studi"). **n-gram language models** — P(wₙ|wₙ₋₁...) with Markov assumption; smoothing (Laplace, backoff) because zero counts kill products; still the right mental model for "the model assigns each sequence a probability." **Classical text classification pipeline** — clean → tokenize → vectorize → linear model: still a competitive baseline the interviewer may ask you to justify.

In [6]:
# n-gram LM with Laplace smoothing, in 15 lines
from collections import defaultdict

text = ("the model trains on data the model evaluates on data "
        "the model predicts the label").split()

bigrams = defaultdict(list)
for w1, w2 in zip(text, text[1:]):
    bigrams[w1].append(w2)
vocab = set(text)

def bigram_prob(w1, w2, alpha=1.0):
    counts = bigrams[w1]
    return (counts.count(w2) + alpha) / (len(counts) + alpha * len(vocab))

print(f"P(data | the)      = {bigram_prob('the', 'data'):.3f}")
print(f"P(model | the)     = {bigram_prob('the', 'model'):.3f}")
print(f"P(zebra | the)     = {bigram_prob('the', 'zebra'):.3f}  <- unseen: smoothing keeps it nonzero")
print("without smoothing, one zero-probability bigram would zero the whole product")

P(data | the)      = 0.083
P(model | the)     = 0.333
P(zebra | the)     = 0.083  <- unseen: smoothing keeps it nonzero
without smoothing, one zero-probability bigram would zero the whole product


## Q9 — Quick-Fire NLP Round

| Question | The 15-second answer |
| --- | --- |
| BLEU/ROUGE? | N-gram overlap vs reference (BLEU: precision-geometric, MT; ROUGE: recall-leaning, summarization); proxy metrics, not semantics — pair with human eval |
| BERT vs GPT architecture? | Bidirectional encoder (mask-and-predict; embeddings for understanding) vs causal decoder (next-token; generation); modern LLMs are decoder stacks |
| Why [CLS] token? | BERT's aggregate pooled representation for classification; fine-tune a head on it |
| What's beam search? | Keep top-b partial sequences at each step vs greedy's 1 — better fluency at generation, unnecessary for short factual answers |
| Subword vs vocab-size tradeoff? | Small vocab → long sequences → more compute per text; large vocab → bigger embedding matrix → more parameters; ~30-50k is the sweet spot |
| Cross-encoder vs bi-encoder? | Bi encodes separately (fast, precomputable, for retrieval); cross encodes jointly (accurate, slow, for reranking) — the standard two-stage stack |
| Multilingual embeddings? | Shared space across languages (LaBSE/multilingual-E5); cross-lingual retrieval works because aligned training |
| Stop-word removal — still a thing? | For linear/TF-IDF models often; for transformers never — they use every token's signal |